In [0]:
CATALOG = "dbw_taxrisk_dev"
T = f"{CATALOG}.config.synthetic_seed"

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {T} (
  country_code STRING, kind STRING, key STRING, text_value STRING,
  num_low DOUBLE, num_high DOUBLE, weight DOUBLE)
""")

def lst(cc, kind, items):
    """Simple list: key = 001, 002... keeps the order."""
    return [(cc, kind, f"{i:03d}", v, None, None, None) for i, v in enumerate(items, 1)]

CO = []
CO += lst("CO", "city", ["Bogotá", "Medellín", "Cali", "Barranquilla", "Cartagena"])
CO += lst("CO", "venue_suffix", ["del Norte", "Verde", "Dorado", "Central", "Antiguo", "Real", "Azul"])
CO += lst("CO", "first_name", ["sofia", "camila", "valentina", "juan", "andres", "laura",
                               "daniela", "mateo", "santiago", "luisa", "paula", "felipe"])
CO += lst("CO", "theme", ["viaja", "come", "vive", "style", "fit", "foodie", "trip", "moda"])

# category: key = category, text_value = venue-name words separated by |, num_low = base ticket
CO += [("CO", "category", k, "|".join(w), float(t), None, None) for k, w, t in [
    ("restaurante", ["La Mesa", "Sabor", "Fogón", "Casa", "Huerta"], 120_000),
    ("café",        ["Grano", "Altura", "Taza", "Tostadero", "Origen"], 30_000),
    ("bar",         ["Barril", "Terraza", "Coctel", "Noche", "Esquina"], 150_000),
    ("hotel",       ["Hacienda", "Boutique", "Mirador", "Plaza", "Reserva"], 600_000),
    ("spa",         ["Armonía", "Calma", "Bruma", "Oasis", "Pétalo"], 300_000),
]]

# image text by category, plus the special cue and the generic organic image
CO += [("CO", "image", k, v, None, None, None) for k, v in [
    ("restaurante", "mesa con platos y copas"),
    ("café", "taza de café y postre en una mesa"),
    ("bar", "cócteles sobre la barra"),
    ("hotel", "habitación con vista a la ciudad"),
    ("spa", "bandeja con productos de spa"),
    ("_cue", "tarjeta de bienvenida del local sobre la mesa"),
    ("_organic", "foto de la ciudad"),
]]

# tiers: followers range and annual declared income range (local currency)
CO += [("CO", "tier_followers", k, None, float(lo), float(hi), float(w)) for k, lo, hi, w in [
    ("nano", 1_000, 10_000, 60), ("micro", 10_000, 100_000, 32), ("macro", 100_000, 1_000_000, 8)]]
CO += [("CO", "tier_income", k, None, float(lo), float(hi), None) for k, lo, hi in [
    ("nano", 6_000_000, 24_000_000), ("micro", 24_000_000, 90_000_000), ("macro", 90_000_000, 400_000_000)]]

# caption templates: key = <name>_<index>
TEMPLATES = {
    "paid": ["Colaboración con {venue} en {city}. Gracias por confiar en mí. {tag}",
             "Anuncio: {venue}, {city}. Los espero. {tag}"],
    "inkind_disclosed": ["Cortesía de {venue}: una experiencia increíble en {city}. {tag}",
                         "Gracias {venue} por la invitación. {tag}"],
    "praise": ["Qué noche en {venue}, {city}. Todo perfecto, gracias equipo!",
               "Domingo especial en {venue}. Lo recomiendo muchísimo",
               "Así se disfruta {city}: {venue} no falla"],
    "organic": ["Planes de {category} en {city} este fin de semana",
                "Mi rutina de la semana en {city}",
                "Probando algo nuevo hoy, ya les cuento"],
}
for name, items in TEMPLATES.items():
    CO += [("CO", "template", f"{name}_{i:03d}", v, None, None, None) for i, v in enumerate(items, 1)]

# idempotent: replace only this country's rows
spark.sql(f"DELETE FROM {T} WHERE country_code = 'CO'")
schema = "country_code string, kind string, key string, text_value string, num_low double, num_high double, weight double"
spark.createDataFrame(CO, schema).write.mode("append").saveAsTable(T)
print(len(CO), "rows loaded for CO")

In [0]:
%sql
SELECT kind, count(*) AS n FROM dbw_taxrisk_dev.config.synthetic_seed
WHERE country_code = 'CO' GROUP BY kind ORDER BY kind

In [0]:
T = "dbw_taxrisk_dev.config.synthetic_seed"
spark.sql(f"ALTER TABLE {T} ADD COLUMNS (seq INT)")

spark.sql(f"UPDATE {T} SET seq = CAST(key AS INT) WHERE kind IN ('city','venue_suffix','first_name','theme')")
spark.sql(f"""UPDATE {T} SET seq = CASE key
    WHEN 'restaurante' THEN 1 WHEN 'café' THEN 2 WHEN 'bar' THEN 3
    WHEN 'hotel' THEN 4 WHEN 'spa' THEN 5 END
    WHERE kind = 'category'""")
spark.sql(f"""UPDATE {T} SET seq = CASE key WHEN 'nano' THEN 1 WHEN 'micro' THEN 2 WHEN 'macro' THEN 3 END
    WHERE kind IN ('tier_followers','tier_income')""")
spark.sql(f"UPDATE {T} SET seq = CAST(right(key, 3) AS INT) WHERE kind = 'template'")
spark.sql(f"UPDATE {T} SET seq = 0 WHERE kind = 'image'")

display(spark.sql(f"SELECT kind, key, seq FROM {T} WHERE country_code='CO' AND kind IN ('category','tier_followers') ORDER BY kind, seq"))
print("filas sin seq:", spark.table(T).filter("seq IS NULL").count())